## 0 Purpose

This script calculates a wide range of spectral indices as well a texture parameters (curated Haralick features) on the basis of Sentinel-2 imagery. (The haralick features performed badly in the randomForest_Training.ipynb and take long to get calculated. So you might want to delete this part). To increase readability, all the calculation takes place in two functions ("2. Haralick function" and "3. Index function") nested into each other.

## 1. Import Packages


In [1]:
#%pip install xarray rioxarray pandas matplotlib rasterio
import geopandas as gpd
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray
from rasterio import rio
from rasterio import features
from rasterio.enums import Resampling
import numpy as np
from rasterio.transform import Affine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops

## 2. Haralick function

Ich will die Texturen verwenden, deshalb hier eine function die die haralick features: contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM' aussrechnet. Diese Funtion wird dann in der eigentlichen Indix-funktion weiter benutzt.

In [2]:
def calculate_texture_map(da, window_size=5, levels=32):
    # 1. Quantize
    d_min, d_max = da.min(), da.max()
    quantized = ((da - d_min) / (d_max - d_min) * (levels - 1)).fillna(0).astype('uint8')

    def _haralick_block(block):
        block_uint = np.nan_to_num(block).astype('uint8')

        glcm = graycomatrix(block_uint,
                            distances=[1],
                            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                            levels=levels,
                            symmetric=True,
                            normed=True)

        properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']

        # FIX 2: Return a numpy array instead of a list
        # This allows NumPy to see the '.ndim' it's looking for
        return np.array([graycoprops(glcm, p).mean() for p in properties])

    # 3. Create windows
    windows = quantized.rolling(x=window_size, y=window_size, center=True).construct(x="wx", y="wy")

    # 4. Apply
    feature_names = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    texture_stack = xr.apply_ufunc(
        _haralick_block,
        windows,
        input_core_dims=[["wy", "wx"]],
        output_core_dims=[["band"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float]
    )

    return texture_stack.assign_coords(band=feature_names)

## 3. Index function
Diese Funktion errechnet diverese Indizes für die Sentinel-2 Daten.

#### Optische Indizes:

ndvi

savi

ndwi

bsi (Bare soil index)

nssi (Non-photosynthetic vegetation soil seperation Index
nach https://www.sciencedirect.com/science/article/pii/S0303243421000684, wurde entwickelt um bare soil und nicht-aktive Vegetation auseinander zu halten


#### Texturen
Auf Basis der calculate_texture_map Funktion werden die Features 'contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM' errechnet.
Zusätzlich habe ich noch die Standardabweichung in einem 5*5 Fenster ausgerechnet.

In [3]:
def new_indices(filepath):

    # IMPORT DATA
    raster = rioxarray.open_rasterio(filepath) # raster öffnen und bänder benennen
    bandnames = ["blue", "green", "red", "rededge1", "rededge2", "rededge3", "nir", "nir_narrow", "swir1", "swir2", "trainclass"]
    raster = raster.assign_coords(band = bandnames)

    print("Raster was imported successfully")
    # SPECTRAL INDICES:

    ndvi = (raster.sel(band="nir") - raster.sel(band="red")) / (raster.sel(band="nir") + raster.sel(band="red"))
    ndvi = ndvi.expand_dims(band=["ndvi"])

    savi = (1.5*(raster.sel(band="nir") - raster.sel(band="red"))) / (raster.sel(band="nir") + raster.sel(band="red") + 0.5) # mit L = 0.5
    savi = savi.expand_dims(band=["savi"])

    ndwi = (raster.sel(band="nir") - raster.sel(band="swir1")) / (raster.sel(band="nir") + raster.sel(band="swir1")) # nach Gao, misst Wasser in Vegetation, heißt manchmal normalized difference moisture index
    ndwi = ndwi.expand_dims(band=["ndwi"])

    bsi = ((raster.sel(band="swir2") + raster.sel(band="red")) - (raster.sel(band="nir")+ raster.sel(band="blue")))  /  ((raster.sel(band="swir2") + raster.sel(band="red")) + (raster.sel(band="nir") + raster.sel(band="blue"))) #bare soil index
    bsi = bsi.expand_dims(band=["bsi"])

    nr = raster.sel(band="nir") / raster.sel(band="red")
    nr = nr.expand_dims(band=["nr"])

    nssi = (raster.sel(band="nir_narrow")-  raster.sel(band="rededge3")) / (raster.sel(band="nir_narrow") + raster.sel(band="rededge3"))
    nssi = nssi.expand_dims(band=["nssi"])
    # Das hier ist der Non-photosynthetic vegetation soil seperation index: https://www.sciencedirect.com/science/article/pii/S0303243421000684. Sollte ja genau das Problem lösen, das disturbed und deadwood verwechselt wird. Nutzt Band 7 und Band8a, ich bin mir nicht komplett sicher ob das diesen Bändern hier entspricht aus deiner Bennenung.

    print("Multispectral indices were calculated successfully")

    # TEXTURE PARAMETERS
    print("Computation of Haralick features started. ")
    layer = raster.sel(band="nir")

        # GREY-LEVEL CO-OCCURENCE MATRIX:
    textures =  calculate_texture_map(layer, window_size=5, levels=32)

    print("Haralick features were calculated successfully")
        # STANDARDABWEICHUNG 4*4:

    std = layer.rolling(x=4, y=4, center=True).std()
    std = std.expand_dims(band=["std"])
    print("Standard deviation was calculated successfully")

    trainclass = raster.sel(band = "trainclass")
    trainclass = trainclass.expand_dims(band=["trainclass"])
    raster = raster.drop_sel(band = "trainclass")

    indices = xr.concat([raster, ndvi, savi, ndwi, bsi, nssi, textures, std, trainclass], dim = "band")
    indices.attrs['long_name'] = [str(b) for b in indices.band.values]
    print("Raster were stacked successfully")

    return indices

## 4. Anwendung der Funktion auf die 4 Datensätze
Gerade die Haralick features dauern sehr lange. Deshalb habe ich hier einen loop die das für 4 Datensätze nacheinander einfach macht.

In [ ]:
input_files = [
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\n22.tif",
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\n23.tif",
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\s20.tif",
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\s22.tif"
]

output_files = [
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n22.tif",
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n23.tif",
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s20.tif",
    r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s22.tif"
]

for in_path, out_path in zip(input_files, output_files):
    print(f"Processing {in_path}...")
    raster = new_indices(in_path)
    raster.rio.to_raster(out_path)
print("Done!")



Processing C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\n22.tif...
Raster was imported successfully
Multispectral indices were calculated successfully
Computation of Haralick features started. 
